# Phase 4: Feature Engineering for Machine Learning

## Objective

The goal of this notebook is to prepare the Monaco 2025 race data for machine learning.

In the previous phases, we collected race data, cleaned lap times, explored race pace, and analyzed tyre degradation. In this phase, we will create additional features that help a machine learning model predict lap time.

The target variable for the model will be:

`LapTimeSeconds`

The final output of this notebook will be a machine-learning-ready dataset saved as:

`data/processed/2025_monaco_ml_dataset.csv`


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROCESSED_DATA_DIR = Path("../data/processed")

clean_laps_path = PROCESSED_DATA_DIR / "2025_monaco_clean_laps.csv"
tyre_analysis_path = PROCESSED_DATA_DIR / "2025_monaco_tyre_analysis_laps.csv"

In [3]:
clean_laps = pd.read_csv(clean_laps_path)
tyre_laps = pd.read_csv(tyre_analysis_path)

print("Clean laps shape:", clean_laps.shape)
print("Tyre analysis laps shape:", tyre_laps.shape)

Clean laps shape: (1423, 33)
Tyre analysis laps shape: (1243, 33)


## Dataset Choice for Machine Learning

For tyre degradation analysis, we used a strict tyre-analysis dataset.

For machine learning, we will start from the broader cleaned lap dataset because the model should learn from more race situations. However, we will still remove rows that are not useful for lap-time prediction, such as missing lap times and extreme abnormal laps.

We will also keep flags like `IsPitLap` and `TrackStatus` so the model can learn that pit laps or non-green-flag laps behave differently.

In [4]:
ml_data = clean_laps.copy()

print("Initial ML data shape:", ml_data.shape)
ml_data.head()

Initial ML data shape: (1423, 33)


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,LapTimeSeconds,IsPitLap
0,0 days 00:57:36.090000,VER,1,0 days 00:01:27.020000,1.0,1.0,NaN,NaN,NaN,0 days 00:00:38.394000,...,0 days 00:56:08.809000,2025-05-25 13:03:09.195,12,4.0,False,NaN,False,False,87.020,False
1,0 days 00:59:22.408000,VER,1,0 days 00:01:46.318000,2.0,1.0,NaN,NaN,0 days 00:00:34.949000,0 days 00:00:45.145000,...,0 days 00:57:36.090000,2025-05-25 13:04:36.476,16,4.0,False,NaN,False,False,106.318,False
2,0 days 01:01:11.223000,VER,1,0 days 00:01:48.815000,3.0,1.0,NaN,NaN,0 days 00:00:32.110000,0 days 00:00:47.430000,...,0 days 00:59:22.408000,2025-05-25 13:06:22.794,6,4.0,False,NaN,False,False,108.815,False
3,0 days 01:02:51.513000,VER,1,0 days 00:01:40.290000,4.0,1.0,NaN,NaN,0 days 00:00:31.006000,0 days 00:00:47.889000,...,0 days 01:01:11.223000,2025-05-25 13:08:11.609,671,4.0,False,NaN,False,False,100.290,False
4,0 days 01:04:12.313000,VER,1,0 days 00:01:20.800000,5.0,1.0,NaN,NaN,0 days 00:00:21.744000,0 days 00:00:37.485000,...,0 days 01:02:51.513000,2025-05-25 13:09:51.899,1,4.0,False,NaN,False,True,80.800,False


In [5]:
ml_data.columns.tolist()

['Time',
 'Driver',
 'DriverNumber',
 'LapTime',
 'LapNumber',
 'Stint',
 'PitOutTime',
 'PitInTime',
 'Sector1Time',
 'Sector2Time',
 'Sector3Time',
 'Sector1SessionTime',
 'Sector2SessionTime',
 'Sector3SessionTime',
 'SpeedI1',
 'SpeedI2',
 'SpeedFL',
 'SpeedST',
 'IsPersonalBest',
 'Compound',
 'TyreLife',
 'FreshTyre',
 'Team',
 'LapStartTime',
 'LapStartDate',
 'TrackStatus',
 'Position',
 'Deleted',
 'DeletedReason',
 'FastF1Generated',
 'IsAccurate',
 'LapTimeSeconds',
 'IsPitLap']

## Basic Cleaning

The model target is `LapTimeSeconds`, so rows with missing lap times cannot be used.

We also convert important columns into numeric format where needed.

In [6]:
# Remove rows with missing target variable
ml_data = ml_data.dropna(subset=["LapTimeSeconds"]).copy()

# Convert important columns to numeric
numeric_cols = [
    "LapNumber",
    "TyreLife",
    "Stint",
    "Position",
    "TrackStatus"
]

for col in numeric_cols:
    if col in ml_data.columns:
        ml_data[col] = pd.to_numeric(ml_data[col], errors="coerce")

print("ML data shape after basic cleaning:", ml_data.shape)

ML data shape after basic cleaning: (1423, 33)


In [7]:
# Remove extreme abnormal lap times
# This keeps the model focused on realistic race-lap prediction.
ml_data = ml_data[
    (ml_data["LapTimeSeconds"] >= 70) &
    (ml_data["LapTimeSeconds"] <= 115)
].copy()

print("ML data shape after lap-time range filter:", ml_data.shape)
ml_data["LapTimeSeconds"].describe()

ML data shape after lap-time range filter: (1421, 33)


count    1421.000000
mean       79.259935
std         6.665921
min        73.221000
25%        76.028000
50%        77.678000
75%        79.404000
max       112.404000
Name: LapTimeSeconds, dtype: float64

## Race Context Features

These features describe where the lap happened in the race and whether it was affected by pit activity.

In [8]:
# Make sure IsPitLap exists
if "IsPitLap" not in ml_data.columns:
    ml_data["IsPitLap"] = ml_data["PitInTime"].notna() | ml_data["PitOutTime"].notna()

# Convert boolean to integer for ML
ml_data["IsPitLap"] = ml_data["IsPitLap"].astype(int)

# Green flag indicator
ml_data["IsGreenFlag"] = (ml_data["TrackStatus"].astype(str) == "1").astype(int)

ml_data[["Driver", "LapNumber", "LapTimeSeconds", "IsPitLap", "TrackStatus", "IsGreenFlag"]].head()

,Driver,LapNumber,LapTimeSeconds,IsPitLap,TrackStatus,IsGreenFlag
0,VER,1.0,87.020,0,12,0
1,VER,2.0,106.318,0,16,0
2,VER,3.0,108.815,0,6,0
3,VER,4.0,100.290,0,671,0
4,VER,5.0,80.800,0,1,1


In [9]:
max_lap = ml_data["LapNumber"].max()

ml_data["RaceProgress"] = ml_data["LapNumber"] / max_lap

ml_data[["LapNumber", "RaceProgress"]].head()

,LapNumber,RaceProgress
0,1.0,0.012821
1,2.0,0.025641
2,3.0,0.038462
3,4.0,0.051282
4,5.0,0.064103


## Tyre and Stint Features

Tyre age and stint information are important for lap-time prediction. We also create a `StintProgress` feature to show how far into a stint a lap occurred.

In [10]:
# Calculate stint start and end lap for each driver/stint
stint_bounds = (
    ml_data
    .groupby(["Driver", "Stint"])
    .agg(
        StintStartLap=("LapNumber", "min"),
        StintEndLap=("LapNumber", "max")
    )
    .reset_index()
)

ml_data = ml_data.merge(
    stint_bounds,
    on=["Driver", "Stint"],
    how="left"
)

ml_data["StintLength"] = ml_data["StintEndLap"] - ml_data["StintStartLap"] + 1

ml_data["StintProgress"] = (
    (ml_data["LapNumber"] - ml_data["StintStartLap"] + 1) / ml_data["StintLength"]
)

ml_data[
    ["Driver", "LapNumber", "Stint", "StintStartLap", "StintEndLap", "StintLength", "StintProgress"]
].head(15)

,Driver,LapNumber,Stint,StintStartLap,StintEndLap,StintLength,StintProgress
0,VER,1.0,1.0,1.0,28.0,28.0,0.035714
1,VER,2.0,1.0,1.0,28.0,28.0,0.071429
2,VER,3.0,1.0,1.0,28.0,28.0,0.107143
3,VER,4.0,1.0,1.0,28.0,28.0,0.142857
4,VER,5.0,1.0,1.0,28.0,28.0,0.178571
5,VER,6.0,1.0,1.0,28.0,28.0,0.214286
6,VER,7.0,1.0,1.0,28.0,28.0,0.250000
7,VER,8.0,1.0,1.0,28.0,28.0,0.285714
8,VER,9.0,1.0,1.0,28.0,28.0,0.321429
9,VER,10.0,1.0,1.0,28.0,28.0,0.357143


In [11]:
# Non-linear tyre effect
ml_data["TyreLifeSquared"] = ml_data["TyreLife"] ** 2

ml_data[["TyreLife", "TyreLifeSquared"]].head()

,TyreLife,TyreLifeSquared
0,1.0,1.0
1,2.0,4.0
2,3.0,9.0
3,4.0,16.0
4,5.0,25.0


## Driver Pace Features

Different drivers and cars have different baseline pace. We create driver-level pace features to help the model understand each driver's normal lap-time range.

In [12]:
# Use non-pit, green-flag laps for baseline driver pace
baseline_laps = ml_data[
    (ml_data["IsPitLap"] == 0) &
    (ml_data["IsGreenFlag"] == 1)
].copy()

driver_median_pace = (
    baseline_laps
    .groupby("Driver")["LapTimeSeconds"]
    .median()
    .reset_index(name="DriverMedianPace")
)

ml_data = ml_data.merge(driver_median_pace, on="Driver", how="left")

ml_data["LapDeltaFromDriverMedian"] = (
    ml_data["LapTimeSeconds"] - ml_data["DriverMedianPace"]
)

ml_data[
    ["Driver", "LapTimeSeconds", "DriverMedianPace", "LapDeltaFromDriverMedian"]
].head()

,Driver,LapTimeSeconds,DriverMedianPace,LapDeltaFromDriverMedian
0,VER,87.020,75.291,11.729
1,VER,106.318,75.291,31.027
2,VER,108.815,75.291,33.524
3,VER,100.290,75.291,24.999
4,VER,80.800,75.291,5.509


### Team median pace

In [13]:
team_median_pace = (
    baseline_laps
    .groupby("Team")["LapTimeSeconds"]
    .median()
    .reset_index(name="TeamMedianPace")
)

ml_data = ml_data.merge(team_median_pace, on="Team", how="left")

ml_data[["Team", "LapTimeSeconds", "TeamMedianPace"]].head()

,Team,LapTimeSeconds,TeamMedianPace
0,Red Bull Racing,87.020,76.639
1,Red Bull Racing,106.318,76.639
2,Red Bull Racing,108.815,76.639
3,Red Bull Racing,100.290,76.639
4,Red Bull Racing,80.800,76.639


## Previous Lap and Rolling Pace Features

Lap time is sequential. A driver's previous lap and recent average pace can be useful for predicting the next lap.

We create:

- `PreviousLapTime`
- `Rolling3LapAvg`
- `Rolling5LapAvg`

In [14]:
ml_data = ml_data.sort_values(["Driver", "LapNumber"]).copy()

ml_data[["Driver", "LapNumber", "LapTimeSeconds"]].head(20)

,Driver,LapNumber,LapTimeSeconds
426,ALB,1.0,91.479
427,ALB,2.0,112.152
428,ALB,3.0,108.346
429,ALB,4.0,96.006
430,ALB,5.0,80.465
431,ALB,6.0,81.426
432,ALB,7.0,81.287
433,ALB,8.0,80.471
434,ALB,9.0,84.151
435,ALB,10.0,81.528


### Previous lap time

In [15]:
ml_data["PreviousLapTime"] = (
    ml_data
    .groupby("Driver")["LapTimeSeconds"]
    .shift(1)
)

ml_data[["Driver", "LapNumber", "LapTimeSeconds", "PreviousLapTime"]].head(20)

,Driver,LapNumber,LapTimeSeconds,PreviousLapTime
426,ALB,1.0,91.479,NaN
427,ALB,2.0,112.152,91.479
428,ALB,3.0,108.346,112.152
429,ALB,4.0,96.006,108.346
430,ALB,5.0,80.465,96.006
431,ALB,6.0,81.426,80.465
432,ALB,7.0,81.287,81.426
433,ALB,8.0,80.471,81.287
434,ALB,9.0,84.151,80.471
435,ALB,10.0,81.528,84.151


### Rolling Averages 

In [17]:
ml_data["Rolling3LapAvg"] = (
    ml_data
    .groupby("Driver")["LapTimeSeconds"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

ml_data["Rolling5LapAvg"] = (
    ml_data
    .groupby("Driver")["LapTimeSeconds"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

ml_data[
    ["Driver", "LapNumber", "LapTimeSeconds", "PreviousLapTime", "Rolling3LapAvg", "Rolling5LapAvg"]
].head(20)

,Driver,LapNumber,LapTimeSeconds,PreviousLapTime,Rolling3LapAvg,Rolling5LapAvg
426,ALB,1.0,91.479,NaN,NaN,NaN
427,ALB,2.0,112.152,91.479,91.479000,91.479000
428,ALB,3.0,108.346,112.152,101.815500,101.815500
429,ALB,4.0,96.006,108.346,103.992333,103.992333
430,ALB,5.0,80.465,96.006,105.501333,101.995750
431,ALB,6.0,81.426,80.465,94.939000,97.689600
432,ALB,7.0,81.287,81.426,85.965667,95.679000
433,ALB,8.0,80.471,81.287,81.059333,89.506000
434,ALB,9.0,84.151,80.471,81.061333,83.931000
435,ALB,10.0,81.528,84.151,81.969667,81.560000


## Weather Features

FastF1 includes weather-related columns such as air temperature, track temperature, humidity, pressure, wind speed, and rainfall.

These features can influence lap time and tyre behavior.

In [18]:
weather_cols = [
    "AirTemp",
    "Humidity",
    "Pressure",
    "Rainfall",
    "TrackTemp",
    "WindDirection",
    "WindSpeed"
]

available_weather_cols = [col for col in weather_cols if col in ml_data.columns]
missing_weather_cols = [col for col in weather_cols if col not in ml_data.columns]

print("Available weather columns:", available_weather_cols)
print("Missing weather columns:", missing_weather_cols)

Available weather columns: []
Missing weather columns: ['AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed']


In [19]:
for col in available_weather_cols:
    if col == "Rainfall":
        ml_data[col] = ml_data[col].astype(int)
    else:
        ml_data[col] = pd.to_numeric(ml_data[col], errors="coerce")

ml_data[available_weather_cols].head()

""
426
427
428
429
430


## Select Final ML Features

The model will use a mix of categorical, numeric, race context, tyre, stint, driver, and weather features.

The target variable is:

`LapTimeSeconds`

In [20]:
categorical_features = [
    "Driver",
    "Team",
    "Compound"
]

numeric_features = [
    "LapNumber",
    "TyreLife",
    "TyreLifeSquared",
    "Stint",
    "Position",
    "TrackStatus",
    "IsPitLap",
    "IsGreenFlag",
    "RaceProgress",
    "StintLength",
    "StintProgress",
    "DriverMedianPace",
    "TeamMedianPace",
    "PreviousLapTime",
    "Rolling3LapAvg",
    "Rolling5LapAvg"
]

# Add weather features if available
numeric_features = numeric_features + available_weather_cols

target = "LapTimeSeconds"

feature_columns = categorical_features + numeric_features

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print("Target:", target)
print("Total features:", len(feature_columns))

Categorical features: ['Driver', 'Team', 'Compound']
Numeric features: ['LapNumber', 'TyreLife', 'TyreLifeSquared', 'Stint', 'Position', 'TrackStatus', 'IsPitLap', 'IsGreenFlag', 'RaceProgress', 'StintLength', 'StintProgress', 'DriverMedianPace', 'TeamMedianPace', 'PreviousLapTime', 'Rolling3LapAvg', 'Rolling5LapAvg']
Target: LapTimeSeconds
Total features: 19


In [21]:
ml_dataset = ml_data[feature_columns + [target]].copy()

print("ML dataset shape before final missing-value check:", ml_dataset.shape)

missing_summary = ml_dataset.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]

ML dataset shape before final missing-value check: (1421, 20)


Rolling5LapAvg     20
Rolling3LapAvg     20
PreviousLapTime    20
dtype: int64

In [22]:
ml_dataset = ml_dataset.dropna().copy()

print("Final ML dataset shape:", ml_dataset.shape)

ml_dataset.head()

Final ML dataset shape: (1401, 20)


,Driver,Team,Compound,LapNumber,TyreLife,TyreLifeSquared,Stint,Position,TrackStatus,IsPitLap,IsGreenFlag,RaceProgress,StintLength,StintProgress,DriverMedianPace,TeamMedianPace,PreviousLapTime,Rolling3LapAvg,Rolling5LapAvg,LapTimeSeconds
427,ALB,Williams,MEDIUM,2.0,6.0,36.0,1.0,10.0,16,0,0,0.025641,32.0,0.06250,77.086,77.075,91.479,91.479000,91.479000,112.152
428,ALB,Williams,MEDIUM,3.0,7.0,49.0,1.0,10.0,6,0,0,0.038462,32.0,0.09375,77.086,77.075,112.152,101.815500,101.815500,108.346
429,ALB,Williams,MEDIUM,4.0,8.0,64.0,1.0,10.0,671,0,0,0.051282,32.0,0.12500,77.086,77.075,108.346,103.992333,103.992333,96.006
430,ALB,Williams,MEDIUM,5.0,9.0,81.0,1.0,10.0,1,0,1,0.064103,32.0,0.15625,77.086,77.075,96.006,105.501333,101.995750,80.465
431,ALB,Williams,MEDIUM,6.0,10.0,100.0,1.0,10.0,1,0,1,0.076923,32.0,0.18750,77.086,77.075,80.465,94.939000,97.689600,81.426


In [23]:
ml_dataset_path = PROCESSED_DATA_DIR / "2025_monaco_ml_dataset.csv"

ml_dataset.to_csv(ml_dataset_path, index=False)

print("Saved ML dataset:")
print(ml_dataset_path)

Saved ML dataset:
../data/processed/2025_monaco_ml_dataset.csv


In [24]:
print("Rows:", ml_dataset.shape[0])
print("Columns:", ml_dataset.shape[1])

print("\nTarget summary:")
print(ml_dataset["LapTimeSeconds"].describe())

print("\nCategorical values:")
for col in categorical_features:
    print(f"\n{col}:")
    print(ml_dataset[col].value_counts().head(10))

Rows: 1401
Columns: 20

Target summary:
count    1401.000000
mean       79.042440
std         6.369163
min        73.221000
25%        76.016000
50%        77.611000
75%        79.313000
max       112.404000
Name: LapTimeSeconds, dtype: float64

Categorical values:

Driver:
Driver
VER    77
PIA    77
NOR    77
LEC    77
HAM    77
HAD    76
OCO    76
LAW    76
HUL    75
COL    75
Name: count, dtype: int64

Team:
Team
Ferrari            154
McLaren            154
Racing Bulls       152
Red Bull Racing    152
Williams           150
Haas F1 Team       150
Mercedes           149
Kick Sauber        149
Aston Martin       110
Alpine              81
Name: count, dtype: int64

Compound:
Compound
HARD      838
MEDIUM    507
SOFT       56
Name: count, dtype: int64


## Phase 4 Summary

In this notebook, we created a machine-learning-ready dataset for Monaco 2025 lap-time prediction.

The target variable is `LapTimeSeconds`.

We engineered features related to race progress, tyre age, stint progress, driver baseline pace, team baseline pace, previous lap time, rolling lap-time averages, pit-lap status, green-flag status, and weather conditions.

The final dataset was saved as:

`data/processed/2025_monaco_ml_dataset.csv`

This dataset will be used in Phase 5 to train and compare lap-time prediction models.